In [26]:
import pandas as pd
import numpy as np
import json
import pickle

df = pd.read_csv('faceit_clean.csv')

separating features  
separating output for classification model and regression model

In [30]:
label_cols = ['team1_win', 'score_diff']
cat_cols = ['map']
num_cols = [c for c in df.columns if c not in label_cols + cat_cols]

X = df[num_cols + cat_cols]
y = df['team1_win']

# Train / Test split

In [31]:
from sklearn.model_selection import train_test_split, cross_val_score

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing Pipeline

In [33]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ]
)

LogisticRegression model

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

log_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipe.fit(X_train, y_train)

y_pred_lr = log_reg_pipe.predict(X_test)
y_proba_lr = log_reg_pipe.predict_proba(X_test)[:, 1]

cv_lr = cross_val_score(log_reg_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(accuracy_score(y_test, y_pred_lr))

0.6866666666666666


RandomForest

In [35]:
from sklearn.ensemble import RandomForestClassifier


In [37]:
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
])

rf_pipe.fit(X_train, y_train)

y_pred_rf = rf_pipe.predict(X_test)
y_proba_rf = rf_pipe.predict_proba(X_test)[:, 1]

cv_rf = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f'Random Forest:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_rf):.3f}')
print(f'  AUC:       {roc_auc_score(y_test, y_proba_rf):.3f}')
print(f'  CV:        {cv_rf.mean():.3f} (+/- {cv_rf.std():.3f})')

Random Forest:
  Accuracy:  0.660
  F1 Score:  0.638
  AUC:       0.739
  CV:        0.673 (+/- 0.027)


GradientBoostingClassifier

In [38]:
from sklearn.ensemble import GradientBoostingClassifier

In [39]:
gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
])

gb_pipe.fit(X_train, y_train)

y_pred_gb = gb_pipe.predict(X_test)
y_proba_gb = gb_pipe.predict_proba(X_test)[:, 1]
cv_gb = cross_val_score(gb_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f'Gradient Boosting:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_gb):.3f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_gb):.3f}')
print(f'  AUC:       {roc_auc_score(y_test, y_proba_gb):.3f}')
print(f'  CV:        {cv_gb.mean():.3f} (+/- {cv_gb.std():.3f})')

Gradient Boosting:
  Accuracy:  0.643
  F1 Score:  0.635
  AUC:       0.740
  CV:        0.639 (+/- 0.029)


Classification model results and comparison

In [40]:
results = {
    'Logistic Regression': {
        'acc': accuracy_score(y_test, y_pred_lr),
    },
    'Random Forest': {
        'acc': accuracy_score(y_test, y_pred_rf),
    },
    'Gradient Boosting': {
        'acc': accuracy_score(y_test, y_pred_gb),
    },
}

for name, r in results.items():
    print(f'{name:<25} {r["acc"]:>10.3f}')

best_name = max(results, key=lambda k: results[k]['acc'])
print(f'\nBest model: {best_name} (accuracy: {results[best_name]["acc"]:.3f})')

Logistic Regression            0.687
Random Forest                  0.660
Gradient Boosting              0.643

Best model: Logistic Regression (accuracy: 0.687)
